In [ ]:
# 3 — декодирование
import re, time, uuid

def _decompress(tag, data):
    if tag == "Z": return zlib.decompress(data)
    if tag == "B": return bz2.decompress(data)
    if tag == "L": return lzma.decompress(data)
    if tag == "R":
        if brotli is None:
            raise RuntimeError("данные сжаты brotli, а библиотека brotli не установлена — %pip install brotli")
        return brotli.decompress(data)
    raise ValueError(f"неизвестный тег сжатия: {tag!r}")

def decode_payload(qr_strings):
    parts, total, prefix = {}, None, None
    for raw in qr_strings:
        raw = raw.strip()
        try:
            head, rest = raw.split("|", 1)
            content_tag, algo_tag = head[0], head[1]
            num_part, chunk = rest.split("|", 1)
            idx_str, total_str = num_part.split("/")
            idx, cur_total = int(idx_str), int(total_str)
        except (ValueError, IndexError):
            raise ValueError(f"не удалось разобрать часть: {raw[:60]!r}")
        cur_prefix = content_tag + algo_tag
        if prefix is None:
            prefix, total = cur_prefix, cur_total
        elif prefix != cur_prefix or total != cur_total:
            raise ValueError(f"части от разных наборов данных: {raw[:60]!r}")
        parts[idx] = chunk

    if total is None:
        raise ValueError("нет ни одной части для декодирования")
    missing = sorted(set(range(1, total + 1)) - set(parts))
    if missing:
        raise ValueError(f"не хватает частей: {missing} из {total}")

    payload_b64 = "".join(parts[i] for i in range(1, total + 1))
    compressed = base64.b64decode(payload_b64)
    content_tag, algo_tag = prefix[0], prefix[1]
    print(f"Обнаружено: тип={'ноутбук' if content_tag=='N' else 'датафрейм'}, сжатие={_ALGO_NAMES.get(algo_tag, algo_tag)}")
    raw_bytes = _decompress(algo_tag, compressed)

    if content_tag == "N":
        text = raw_bytes.decode("utf-8")
        cells = []
        pattern = re.compile(r'### CELL (\d+) \[(\w+)\] ###\n')
        matches = list(pattern.finditer(text))
        for i, m in enumerate(matches):
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            source = text[start:end]
            if source.endswith("\n"):
                source = source[:-1]
            cell = {"cell_type": m.group(2), "id": uuid.uuid4().hex[:8], "metadata": {}, "source": source}
            if cell["cell_type"] == "code":
                cell["outputs"] = []
                cell["execution_count"] = None
            cells.append(cell)
        nb = {"cells": cells, "metadata": {}, "nbformat": 4, "nbformat_minor": 5}
        os.makedirs("qr_restored", exist_ok=True)
        out_path = os.path.join("qr_restored", f"restored_{int(time.time())}.ipynb")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(nb, f, ensure_ascii=False, indent=1)
        print(f"Восстановлено ячеек: {len(cells)}. Сохранено: {out_path}")
        return nb
    elif content_tag == "D":
        df = pd.read_csv(io.StringIO(raw_bytes.decode("utf-8")))
        print(f"Восстановлен датафрейм: {df.shape[0]} строк, {df.shape[1]} колонок")
        return df
    raise ValueError(f"неизвестный тип содержимого: {content_tag!r}")


qr_strings = [
    "NL|001/003|...",
    "NL|002/003|...",
    "NL|003/003|...",
]

result = decode_payload(qr_strings)

In [8]:
# 3 — декодирование
import re, time, uuid, base64, lzma, zlib, bz2
try:
    import brotli
except ImportError:
    brotli = None

_ALGO_NAMES = {
    "Z": "zlib",
    "B": "bz2",
    "L": "lzma",
    "R": "brotli",
}

def _decompress(tag, data):
    if tag == "Z": return zlib.decompress(data)
    if tag == "B": return bz2.decompress(data)
    if tag == "L": return lzma.decompress(data)
    if tag == "R":
        if brotli is None:
            raise RuntimeError("данные сжаты brotli, а библиотека brotli не установлена — %pip install brotli")
        return brotli.decompress(data)
    raise ValueError(f"неизвестный тег сжатия: {tag!r}")

def decode_payload(qr_strings):
    parts, total, prefix = {}, None, None
    for raw in qr_strings:
        raw = raw.strip()
        try:
            head, rest = raw.split("|", 1)
            content_tag, algo_tag = head[0], head[1]
            num_part, chunk = rest.split("|", 1)
            idx_str, total_str = num_part.split("/")
            idx, cur_total = int(idx_str), int(total_str)
        except (ValueError, IndexError):
            raise ValueError(f"не удалось разобрать часть: {raw[:60]!r}")
        cur_prefix = content_tag + algo_tag
        if prefix is None:
            prefix, total = cur_prefix, cur_total
        elif prefix != cur_prefix or total != cur_total:
            raise ValueError(f"части от разных наборов данных: {raw[:60]!r}")
        parts[idx] = chunk

    if total is None:
        raise ValueError("нет ни одной части для декодирования")
    missing = sorted(set(range(1, total + 1)) - set(parts))
    if missing:
        raise ValueError(f"не хватает частей: {missing} из {total}")

    payload_b64 = "".join(parts[i] for i in range(1, total + 1))
    compressed = base64.b64decode(payload_b64)
    content_tag, algo_tag = prefix[0], prefix[1]
    print(f"Обнаружено: тип={'ноутбук' if content_tag=='N' else 'датафрейм'}, сжатие={_ALGO_NAMES.get(algo_tag, algo_tag)}")
    raw_bytes = _decompress(algo_tag, compressed)

    if content_tag == "N":
        text = raw_bytes.decode("utf-8")
        cells = []
        pattern = re.compile(r'### CELL (\d+) \[(\w+)\] ###\n')
        matches = list(pattern.finditer(text))
        for i, m in enumerate(matches):
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            source = text[start:end]
            if source.endswith("\n"):
                source = source[:-1]
            cell = {"cell_type": m.group(2), "id": uuid.uuid4().hex[:8], "metadata": {}, "source": source}
            if cell["cell_type"] == "code":
                cell["outputs"] = []
                cell["execution_count"] = None
            cells.append(cell)
        nb = {"cells": cells, "metadata": {}, "nbformat": 4, "nbformat_minor": 5}
        os.makedirs("qr_restored", exist_ok=True)
        out_path = os.path.join("qr_restored", f"restored_{int(time.time())}.ipynb")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(nb, f, ensure_ascii=False, indent=1)
        print(f"Восстановлено ячеек: {len(cells)}. Сохранено: {out_path}")
        return nb
    elif content_tag == "D":
        df = pd.read_csv(io.StringIO(raw_bytes.decode("utf-8")))
        print(f"Восстановлен датафрейм: {df.shape[0]} строк, {df.shape[1]} колонок")
        return df
    raise ValueError(f"неизвестный тип содержимого: {content_tag!r}")


qr_strings = [
"NL|001/001|/Td6WFoAAATm1rRGAgAhARwAAAAQz1jM4LYcHUhdABHoBAhkUnO+bYNjSEZg41SBT6wY2yARUV4mZM0nS36y"
]

result = decode_payload(qr_strings)

Обнаружено: тип=ноутбук, сжатие=lzma


LZMAError: Compressed data ended before the end-of-stream marker was reached

In [11]:
# 3 — декодирование
import re, time, uuid, base64, lzma, zlib, bz2, os, json, io
try:
    import brotli
except ImportError:
    brotli = None

_ALGO_NAMES = {
    "Z": "zlib",
    "B": "bz2",
    "L": "lzma",
    "R": "brotli",
}

def _decompress(tag, data):
    if tag == "Z": return zlib.decompress(data)
    if tag == "B": return bz2.decompress(data)
    if tag == "L": return lzma.decompress(data)
    if tag == "R":
        if brotli is None:
            raise RuntimeError("данные сжаты brotli, а библиотека brotli не установлена — %pip install brotli")
        return brotli.decompress(data)
    raise ValueError(f"неизвестный тег сжатия: {tag!r}")

def decode_payload(qr_strings):
    parts, total, prefix = {}, None, None
    for raw in qr_strings:
        raw = raw.strip()
        try:
            head, rest = raw.split("|", 1)
            content_tag, algo_tag = head[0], head[1]
            num_part, chunk = rest.split("|", 1)
            idx_str, total_str = num_part.split("/")
            idx, cur_total = int(idx_str), int(total_str)
        except (ValueError, IndexError):
            raise ValueError(f"не удалось разобрать часть: {raw[:60]!r}")
        cur_prefix = content_tag + algo_tag
        if prefix is None:
            prefix, total = cur_prefix, cur_total
        elif prefix != cur_prefix or total != cur_total:
            raise ValueError(f"части от разных наборов данных: {raw[:60]!r}")
        parts[idx] = chunk

    if total is None:
        raise ValueError("нет ни одной части для декодирования")
    missing = sorted(set(range(1, total + 1)) - set(parts))
    if missing:
        raise ValueError(f"не хватает частей: {missing} из {total}")

    payload_b64 = "".join(parts[i] for i in range(1, total + 1))
    compressed = base64.b64decode(payload_b64)
    content_tag, algo_tag = prefix[0], prefix[1]
    print(f"Обнаружено: тип={'ноутбук' if content_tag=='N' else 'датафрейм'}, сжатие={_ALGO_NAMES.get(algo_tag, algo_tag)}")
    raw_bytes = _decompress(algo_tag, compressed)

    if content_tag == "N":
        text = raw_bytes.decode("utf-8")
        cells = []
        pattern = re.compile(r'### CELL (\d+) \[(\w+)\] ###\n')
        matches = list(pattern.finditer(text))
        for i, m in enumerate(matches):
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            source = text[start:end]
            if source.endswith("\n"):
                source = source[:-1]
            cell = {"cell_type": m.group(2), "id": uuid.uuid4().hex[:8], "metadata": {}, "source": source}
            if cell["cell_type"] == "code":
                cell["outputs"] = []
                cell["execution_count"] = None
            cells.append(cell)
        nb = {"cells": cells, "metadata": {}, "nbformat": 4, "nbformat_minor": 5}
        os.makedirs("qr_restored", exist_ok=True)
        out_path = os.path.join("qr_restored", f"restored_{int(time.time())}.ipynb")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(nb, f, ensure_ascii=False, indent=1)
        print(f"Восстановлено ячеек: {len(cells)}. Сохранено: {out_path}")
        return nb
    elif content_tag == "D":
        df = pd.read_csv(io.StringIO(raw_bytes.decode("utf-8")))
        print(f"Восстановлен датафрейм: {df.shape[0]} строк, {df.shape[1]} колонок")
        return df
    raise ValueError(f"неизвестный тип содержимого: {content_tag!r}")


qr_strings = [
"NL|001/001|/Td6WFoAAATm1rRGAgAhARwAAAAQz1jM4LYcHUhdABHoBAhkUnO+bYNjSEZg41SBT6wY2yARUV4mZM0nS36yLF4Pvcgon8MKZWccSQRMYk3ZFeI3pLpdXcz/VtRPTPXuuIhaZdfrfDNhU+mFAtkrdU9l60OGHAnKb1HqpSwHCNHGcfZrgrvh5uOsrl2DSxLy4rqBqAr5FwGoakNeA9iFOZ6QtRevm3UG2Sn3wk+DWbErL5WOEyFQ0Gh2lBKYDK0cdlIQal+0kkWqIOOi7OBNRL4ZEXGyzeLK8PdWCEJ5z1MtZMZVVU45gKXMfkIYIYu6rYGITe6YouuwuJhiYsIHUinR2KqpSOOcYRtuPvjPybwd1vScMW0QG9RzcgSo242f13oY6ghjHjiELrQDQsc7x7QcACv+/CXOUxyJLouyLnzcdzkYLGPWVTqHqQrHW5Jns5EOr03b/kNpxdEBLkkjKp1+DuMJpuM4+8Mrq9a3tinlh8CwiI7Ne4csuPSouWJxU0c6NmWtbCKq1TNxZxxpE9s0a95psaNQL+5CIcLcr2kE2M8/aPBsMYU42VTJSOoGG05IkP9Ysu3wLjBXZrFrFXyqgHiNVK3fWz8OpGLAIcmJHpsqn3Ud5adixTYGc1j1f+6JqjJGeewbPSX4U9iVn9+XcfIQwvlINJaTyMntnImsjsV20At4/wlGxbcETYOcFcJH+aMzLMIweeyUxJh/Jx7OipXInTog/tsDqAAbvB07jWKv46BVq0wMp88ZqsLd6Sli3NCBs59hOY27/5m9GAsJy4THPXQ83NmiZ++WXPw5KrHKTwRQZ48IdMpCVoQvqURUSYvLAI8AgwJrt2KSeXNvMIeQQnzwyudyWUE3Qk4Eh10W2UYMio5iEd4Gmk1vS5wERE8z7EERMm3g1bmA0gEyUhHnO4tFDVuEQ7QHYOAlg2a0sAmmHpxRZNyBS4t2YMVuluM1DOLMYozPbql2YB4Q9SQ6SDyN+wwm4QnyqLK4xnj0k175KQSuSC7vvRjVFUb5yr5ztDuGbbnatqDT1KdH7XMfQ/By9zl00UtOaMWVPwApSyFdm7cW0Suu/gqRRlcLboe2PrB90JafqEXVlabYHCHZ4QK0PhFi9QlpdodMK3D8UjTr27a2yJeva8Wude+WbeuCtqV2LTgDo+9HAgCc3nIYvPpizA2OgRxdOyxkE/Mpojw2xWBHdOGvQUUfeBjt/8erppOkXOENpeVXb5/DU5EFSAEIkaXUBm4OBF3+WRCdy1w1+QCJ90VjVYfJ0hYxLXNlOtN/sIFlI7937t+27ASnSRAebo5V8o5QthuLcjbyFF/k2RUsKMw1KuwTs8MpL9+23ism5bDNpz1AhBYvB4jVMzMDjZKCh12tT6U2wB2BpnR7Ralufs6JQFRQUqoAFpTQ863ISg8sEKorKuU4pM40pCllrvp9InnRspiQ6n1YI1RSuOupjxeLdSRCmJ3tKiMZ6XMb49D62bObujfh/3rq+gF0Fl7gHIkZQSOgN8XqD4IpkAitfGbPKizZxr4kd3OI6bHiJc6CT/orKRJd2jqIR2P+gSG4JXyXzw1XTkxSbFYuUBi2ySIUpmC1hsXutOzFn/h+XWAELH+JV78P+sx7HYLh4T9Cg7IftDlYo5C4ybLWYD0PxHcSQFlJIWmM3yKy7OZFm+YY3DT5n+Qjy6ID/WCD7vY5B94VTJRNeDT/oQ0OIdAPqpau08UEA8/8VfOS0WsxwGAErg9utupmFXkWvFw4+h+0DWUv4b2mFJ7E6FlKgBJoLQ1ksvh6eZEtQygy1e+mvaBetWrVTtzG6UQtvS8i6bvXhGGV8/0SJf0LLxXuf0mPtW6QtDwTzXpCtGktQgSn2yqndND0rKMqRvFbOj2ol/3lIl8fXqu26uSMZTuCHBU7Xc8l19IR7nRC92nXI0NMyny4WcqB6r5BrYu/frYy0VEXDztmJ1/WalOSjFC89+B97XFkcYH+bgR7zeWE83qtZE8IlI4SROwEnK9tA0qddHE1fVPVJ3alkWXBU35xG9Tex9pPGGSggjaPrDRusa2rZbGN8OKbqagpkapbeaXWEV6ltRE73xB0VKCooIRUOcrC9IJG+U7dh40WRIHFD3MOM7gXP0Qj9lZrdj49t3kvVyeckQPScExhkiu4GoHh9lHC3wAAhFzWO1EMOo/fYCuClPviCA09uv9JXgmY02n8LxcXBpkczSAJDqnJtjS8ij5YwqtB3Zb9NucEUG/+8JVISVYW8NOUhwOXU/AoDCE6SYnmfWo32gs8G4Hb0csRxqPndFiJKd0TC0HE3CGOzO6KVS/XMLxsAY539zfdUsfCBTcm18UevCU4gQs6UL22dBY/fxrfb8mD75E8LdPq7dyp8PoPOciFk2wcNdIR1m1bq29KQw+Fq2YRn4f6hzkP2jR/B0q3vjjbzMKk2a5PCRt+9ZyzBOz14oM0urAoKAZglZDWe5i+USqqxtLwWrzwBXJOiUfcXYAzLvLbq5q91E9VFwXq9M24XTPLVPalZfErAxfVlVC0UXTT5a/pEPdsV6Xf+DpM3/fVmXNfVh6nscAPD/gBacJAE9E7ZAtMANkdMDaqofNkRGpKBSQb4mogRhF9ICPfadWfWobwvF59ZodjBZINh5LJcC15ohaHwpImAn1f8vsd96FN09eUE2jku6ojNIUku0C1Q3HRpyVzBuxz92CjhhLOGV5WPOa4scfbxM3M30irG8BW+fvCW/OAKt+XuNTwx7ZYJOAnBzaVC8VHHJhx4ADdoK+5qe8iVYdHV8yGUOG9tfLvVtgtqq0Zc8fs/UDU9KpJEqK9Y2wZ4WFbYfwd/Zn0PwBXJgIsKawrtLDmQ6pF9R+lVw3+9ENr8tHZfgA4IKFh89xPyB0Q6hV7UOWnwdplBpii1encfY65O18m7DgJSwwp7G1GwgQUgNhPgcfLqCwytuCDYlmqwMFTlEt5tScC5vE6Glx3Of8MlndxiIAhWw5Hge09Zo1Mx43JcdhNKsEVbH6Rhg17VBWOjfQrghPL5Hgd/wXs7MzzdeyAP00jSL//K7ngTzgfu9mj48zARDNCg1xIlW0otQHW8oC8BPirlzmHEeUHUci8b6yLF3gUtgLwaPK1rb5w/cKt3M0KdedJpw/MuHT5dry0XxFpDCYl2X3Klw7hr9I5dXSrnekZEbloy6sZYjLK8GLdwaewarfZQUGmkXIv0Q/0MT4jxNIh0hN6Zzlh2jFd3adk2dsXZq0Lf1Fj1xt4tnA8SIE6XRt98EmaxSYe8BgxqGAvl1uOPDRFKDHcQhgKhmJVMwRfky6g5hPMf/Ecpy0qheaJNmgee+JTyqEWSwTcj6E35SNiGNtZML1WPojox1mqqwsrbHT0NWqzVIaLlIY6IbZtWmGyq5O94SifU6ZIl8nlAF32gNkA4jmwYPjyfvurTCoYwmGJJGLyPs0+nbc0AI5RJU/NGklwyNxbzLByswFeJEI2RIAzFZvGG6Rr7BMCsuzYbzSr16iPWNTxna+lyaokz7wnl6pt5GF2fJBaG4PWl8H4nxDrbv0AKt6ojNS0rk9TNKfyqcxAkebaIp14N3MGJaaFryKkySQKVOQlAjSwRA1JA0lPfp3F4b0rcs3V90Xfh6mFbZiTSQzKFndmL0g8PLIzexJ3G9/ZS4dKUiptzRt9dXZjN9AWkJicz/LG0HNMevwlqrk5LTtcrWyY1Wh16C8va+9cq8CHTW8wBmx2HOihVL/YJz3OuZfJYzWiDjl7xRhvhlrtPdsf3cjfozHX1TM0ffZXfpuKdwYJAtiMGtcp3GWcdNAt4pg822gM9TXFyJHdzt/bN1g25KuGN85SIZhzKHbgQhRp9KI7Yz48t4kjvGX6fhAlpXLDNMqJW2oSmq8e2Noq8tsTtBWsmu3w6yGxVc5rTxcSYQyZTZUrIwmpXub9x40vUXZcdHmUdSDvMA1QpfbbkP/YSMJlzlyuBcb0bhO4c9+XJb9FTb0NTOkbvPu0gU2fuwfWUnQ9GOUXILtlYoDB6Y7uncuzqoiSGy3eWu5AqG8p6TQUS3Qo5256ZonsPVL2RBrJZteSnje9BXjcelWQei+ILuu6xpunzotUk5Y28MCrYPAWoI4NaVe+lHfqoW0aH9EyGopT46omoXgnlgCSO66ZMa/hcG/mXyN687PClKomTAx4iWECTWnMk9xA+oRjVZKvxF1k8aD7eqUJpzdUbGV34ULt06vbeI1iILYWdxnUExfeqyJKSBhoSAQev1PBOh1qbU1fhgcqSlTfn5fHkANKG4T6Ne+PmNwbvmjosbsF7uFn1ecmX7UX3XT1qW306pA44vKLsE9WqsDA1S46alnJv1sxynJFdL4Qge62n9jXB0hzC/BqX2/grNjcrbSoZWemWMFuRYFUKzhqx3WbSjj4ruMbLbF6Gt9EhldGIA6HgsYXbiHNLQTxyaiP+6bfjrxCQzP3lUvv13stS8wLkX5ZUKChGH2bdlrZNe2fBE/xgxUUk+Ir8smn6zzJ2O5I0Tzj/rVn8usBGUOhM/eX6u1cHpMjSw63CIKB6VTKmQqxCMyUHn13TjuAIwGQotDo6O18kH55lyIVAWxpwA4xn815xrn22SPiSu5tHSWsxoMVuGVN1ohFMOWQIBMIGIZ4/NSQ0cWhHUZzTRcii8gjKMzEHe9eRruoxb2tcVfyb4JSpv+pAaqRJVlPwYZ7FI0eMkG9EnF70Auctk7uHNbbbfIUekdV9cNwjbyiJWrAB0wWObsEbK/1iVxmWravMmPsysBJXCoBasWCfBpvREMq6bn97R5il9Q9/kzYj40GGTOBcn8fZGcLWn+0iwnixB+iYjE/jLSUGQeHVYQl4rfVusP+QlvTviPbmU+cmM2YSByZozO3PcCHAMW0DcJeQfGM8yZbUkD12uPR+Me8arGyb5XXARpJH4ZbkmxaF7dYaER2iD1gMzhFT8oDl1e0XhNurAeVuCdZOqGw3FDfiq6cYJGyfUBs/9T2OwnN3mnYVbEEKnKE1oLlg9yCriyewoxncYo6CZzwfX7XAyBVz4BNeV7DojxsLSqj5gtAZOAlvRXKE7fwZMUcWrceARjBf+MF0RwAKOoM+QjFBnDunBUa9lSORF6Mgx/Tvyy7WI6TGwHjJ8syrLGQ25j0KtHph20+X+/n60noSfDMfhpyntkPwIZ/8AE6Cq+ky4sTUI+PyqqkIg1wtcb/qJeIouLQjxXA5/f3RkFtOcAmdBhJMiNr03pfIIJidXRYFHBUqL2MwiSTvLHLWJQ0h4/7C/bY+bjhzhkbyE+p3ZMvlpnYJgdoVYtQZrEAs/0EefwrPQNsjmflHORD4RcpMHbnYkEBgnF6f2gQ+FFAtvkRUAKapkJcMEc2YfLQIybVJ5BUEBBbirccID2KuE9drzXkteRo4N8/oCunhfH9ppvG3/FFCloiN6B+bWkZzAcZ9Ihb/GzBC98KDSIdzFtu4RiXYglVJrKZpv87qaM+gXBBszL0SrgkFZShoLqZnedvfY1/I3B2w+J6XZJp5YWArSPqPoMhq/9yFOhUvWZoMXfL5q6bulbtGGCU+0nrT1WJu8/6K9aFJZx+2tQZcECAz/RASNTZNj/J0JJQqSxCiQaiVHpxvhseLxMPz1iousVnb1M8VXeWWqG3YW6gnqMO/7+9lK0xlvPNDtNT0aRNLL1Agiu7uYUD/j/gPsTPlWA5daoWcqVJuRtsEBdE+A/2P4Ol2Nhkvd33J9YybP6xqP7DrPxgRA32JfTHJ+lvqZpAbrBfb6cu8VHbVoXXBWgDOJmnyML0x6T7cV61RxGtINZ9CrkIbmaxTUN4G7tPLmbQNzsKmMirGFY8LUs1U2Yz4GIUknlv5FppF2vJ851/+aJHcRGwD2oI4u77hofb4RwNBWdYAGetSK7qkTuFhyk5SP0UfJGkkUpjfzJp6HzTe8mXR4PWJW9Pkq+BZFrMbcnDEataiAMhuXQ8lmLPw0N8oZ8BI7XgdpsfcqAM0CwgtC6URs5d9yDEB7w4zDMvMy0QYvekCBeP6QumHfw0Wj0IqJrxFNoVGYANia/u2jr9P+u1jOK+NIu4XWM6Fk2qb1Zz3mUcS7/gw6qK9tdExiWy8mbNhuOxHM3pD3FxSqjoHAD+J92Out/7zj5CnNMU6yNXGopPIWVjPw7ZlQQXcMjjHfbdutHTDkUhtSMEbS6A0KP41UHaOlu5CgBXS8Ew0xYRzkihqnEQM5UtvoUjR9FI7XQXQ8BMRfIFCq6Y1AlatH4+RS9Uf9UGYHSsUgeg1dXMjo0UGpv+dIKUpHELeoPSaEvIvFqgeIEBM/WWwRhb2nEjgmVb3t/n2R6Kp6n6OxrSaJzWRgrLAvIvuhjB4yvC7AXsNdMsTILIrA+Wo5+XKDIxAXrXNx6XHACk9dfRdba6FJB2p5fNiw+aOCuruZoD+I/tFwwPhWnm/CO+BGk20fBundzSh4n2qoGxCNmZPHaz7zDAK38q9jGkRcZuYIkuPQtzq/CiWV2OMvazbcfA3IsTNlbDLJ8NWiD+9sB0HM64GfWzJnDNmKOW/3VBd+oK1dKURhmfz4ooAs84Wi66KJeAFq0dvSNuhdH6m/Y+9d+97yIxCZXMhTZHIcT1DAFtf6f3dxaBjfC3nRi/HoVfochpcoGrdvBk9uYhB5jsDs0i08a/M+NeGcgTopqN+vqUnZfScgGyIsfvlku+Z3Sv00tgHePwmXogfBLYq7fM23bqFcz8FJFC1EgA3xG2fljdB6NAWi1aTb+8k3/4tUlPcmRdAlnAzRY/6RJ0n+FehCcED9N0vNAzX0JZms2c964tAxzo5TAv+7s6t2nLg1Qzh1pXVbAi5PepQmfttSkdAmW0GCyD+g/m0J6FpTYHLSXoewCZWqBWOOwLBbzB5KQFdKAqgE8zmQYuHoGmeq/8WJFXSCxjavBj5+/fGPDJ3W3kNVwH/AckYltlaRQLUUPQ+vTqcBSUbVljuDCXQ4G741bKZJjoKt9lJGxQbjsG+wLoo1efZa7D0CsDRmaAz+y0Z4lFm/Jf2OmNAiABFsPob8CqnVCF4S1OmN2qSY540mYSWKbI5IG/RwLb8osH4VMXrJY1VEHGcdzjD7PXsIQ65pOe4UnuN9VXMhHJfpNvZLYnWK18soVdGSSewrk3Pcd2wCOrwnDVn0wuMDToK9r5AZmoK9V5zS+MpAblbMMKRxQR6vkvF87BnHSUdkR1Ee89cE7WRuUOMqrArcrsqZn4Op1f8Dmk4tzUVyC5ml5onUJ+WQtBGzeHLVM8AFJiDDvHkO5MlrmBF3TxvTeaEf0G/clHbDXEYFcst+IdqQfydpnjyuZAZv8PtfKYWLOZamN7KEU+Po09QXWxJNxWDc3i8tFTMb3eNSRRySVDkqTEfO/My6k0D0xaISOVFIRkk8nzT5PDoTl/HPbrd0MpqUB83gaMrTwzSLxpDMNn8CQYkmhinltp0WcLxvVVakNjIch1Fg1JloWDe7IbcACXoW9vzV9cXjDGivdNsjQoxZi3upNyhOnC7oa5ZKPZLyxUqvUaaqE9dQvtQIA7ME50CgS6LmTa25Nonx1ifq6+rvABuElBbOLReMIIhbM5EdV9QisinYyJ6POgexEdcrezu+vsqIEz6qI80+DVagac3eoPCtK1PTIqoXoTb3jPWD+4BOAa7C4AwNNQDAlhHYhRduWf2k7QiNqW/boIbLkrhj2FuxsERcKyNltfUud2DgBV8IC3u5cPv52nWRKa0uCviwl4/3BQwA/NocjpsPYCvPmrmcW/qlXSyUwriI6j/dZkx0YhtqQ9h5y63PwGwRwFkV+ltb7ep0mj9gKlNIevhcM0P55Mv+gbAI0o9n7pqa2r45X8yZOU7dYeXhTzoVGM5YuFIFgXOqT4Is+MmywZCQxKUs0OGVwgeAz0QT6UmLKbQ2eRvu8pj/f9CD7sar5cVV+X2dzuK+5Ybv8Hnahn2zKhqi2l6e0oEfNXYV7dOKbJk2qDIjch0YkvHS/+spbS+h3cklZOp4H2IyNzj8y78iwJQqHp4q9XWMIbD1fY6mdTl21Qmv9TBw7bYT5kT0YcaDd0DNeiDlmGQb5iJOKQpG9UF/5zmgcx+3MQjPyKTkEOAgq+fvs+txW7iYnvhLOAiPB0jrQqFLiHG9yUXb7jDR7WN0NUb/vjKoYag2hLET5wqqyFzZm9t7/vJZh+5ZmV6sooIAcEUs6/BgYVjl1jwsn+8WVws3g4XbJJ312XcIGckUmNPKehb+s8cYuFmM3r+79MAUlerSHBo61M7lBdCtXWlSVP9o0+TWKTClfKeDAY6R0M50cUuYpPfzxtM2rwsWlOa10GtuBLYQaSHxaIeIjd/oGc/P5bd5bDxE5V+06ddvvRfbSMHwYGh+ezRnP4tUCLgmj2mEH5EV5+0C2GX2v6HLSGa9tHUEcLRGY3VR9BOxIbPf5+Lq3Mlc1ukFRtwrdA2C05IcT1lZgRCzCCqcBAnwEi6gy9yt9cZPVuBWdHoVQae0gXK9icwju7VDUTPUG9XTJ/4TPULVirKMi6xbhgsMw5JwsNzd9UxovKFvREz5ezLaXSlywQFhyCyGJvCjKjSwO6mgFIfHR9PrJ+D05k4oEHUxRHDpvVfJEoKE7SkUDXMSu/TCZyGXEOgZZrYIdoZrJDAKxcLyfYati5vgxtGds5XFyY/M3lDGfGdsadXIDMa5QqE81G5D7i8TVVPHXx7oQ7PVTJ2lZoM/BSHoVtzJI9g9PxH3HyQYyzG1MALL4VojZJF3V+mGqjXKsGXNqUn2+Hwo7P4EctbJ/ZQN5G2bHhG9KGNtggjs3KVW7Pn7LR/yfR+yPPZkNmr7vWhZ3amBTI1bQojjCV6D07yt+TRaZScGUJLdYqFq5mvfBYuh0SB98NYXLeInZamjm9l/E0/2mQAvjS5DxpWt+g7De2HOcKMKa4tvIJXBAfhHzLeaw9Y6qesvtgldB5reygOf2WF6MmY2SuUZOCx15zRuJU1jMQmbW+8JRjtXG7PgXFm0lxT7H7GFnJ/EvZj+9oUgSkqU68t3SgXfpb4GC/q/RWejpCHNESIINyNXb+P1QGpaHdV7swYdHqc7v6vIEderfPzj8u02DD/NQ+NU3fx2lS6fCWGRMYzXV3RMktWhxQ1IouKcX+umpKEz8YyEHLdpD0SrQOYNAKdraJ3eZVkx/pSq0tOMQmOyczBDfNQu1v2+Bufk3XffOeFL4EAuHpAKkXo3HjH7+bQEHsOmldiLVYI5QM6d+4J8vnwwqOq1wVVgDQvv4hOpcEiNCEoqUE6+E/p9SdnpJ++p+oSYmi/FA82kVlJWHDDW/Rh6Y2d1PbBHdUmi+Kb50wXLB+ALEuGahmu5PWOVeTWrx3XPnGTrWj9FzvmZWQ7Dgo5//sAQH/TWcA+XVZAFrPZ+WgyttnjzxdfecXnN7CSJwJyeSp7w5CpiuJHcFgZTvENkDUoTptI8IKCjJviH2dy8VHosZqHWEdTcdAheZtWDgMRj6JV+9ax43FU+GYHu7YLowN4kuSoYO05xewB+C6C56oL6MmYGMbw4lwoubt8HLkd0bqpsN4DAbOrf7nX+n2W4X81hLIhrFymw+aTBOMTBM7pAoOZIe0U8G27MhmiPb7MCFOV6UMgwhJv1IBFZv6HInPnpMBUDNQOe0dLv72g9LF6SK3NKiXk6jkni/WBhMz63DK7a3Y071vBgfrrG+tNBXCuYdqsJYpn1LRe84kHrcYGwb5uvaGms9tA84kbZ1jErDLKmByODXFvQxDOYhRU0eeqlYuASHb9OjygMvzLOaJC7xV2Aj0ugY1uqy0XONnLdlXdhx4CR8+k3DzGvGZXXIp1a1ncSgxNsmijUcHJOgPHag4TGUyQwlQMtKK+F+NVJqSajfAHhhLA4ehEw3a9xTNf4vAmr5L//LdujG0rRe8QK/PmQ3kca61Si+0KsplgQFJkBdzNSxjxlG0H5jBgf8VjqoHRru5bsOKd0/M11YXQkqPwvlwnEqYLiHJ/XmnKHJZQ4aZz5C6ih9poaq7/xk9z4Mm9x4c78r+B6pkxew7JXt+APgf8FInGVSvAAHkOp3sAgBRX/V4scRn+wIAAAAABFla"
]

result = decode_payload(qr_strings)

Обнаружено: тип=ноутбук, сжатие=lzma
Восстановлено ячеек: 80. Сохранено: qr_restored\restored_1789035051.ipynb


NL|001/001|/Td6WFoAAATm1rRGAgAhARwAAAAQz1jM4LYcHUhdABHoBAhkUnO+bYNjSEZg41SBT6wY2yARUV4mZM0nS36yLF4Pvcgon8MKZWccSQRMYk3ZFeI3pLpdXcz/VtRPTPXuuIhaZdfrfDNhU+mFAtkrdU9l60OGHAnKb1HqpSwHCNHGcfZrgrvh5uOsrl2DSxLy4rqBqAr5FwGoakNeA9iFOZ6QtRevm3UG2Sn3wk+DWbErL5WOEyFQ0Gh2lBKYDK0cdlIQal+0kkWqIOOi7OBNRL4ZEXGyzeLK8PdWCEJ5z1MtZMZVVU45gKXMfkIYIYu6rYGITe6YouuwuJhiYsIHUinR2KqpSOOcYRtuPvjPybwd1vScMW0QG9RzcgSo242f13oY6ghjHjiELrQDQsc7x7QcACv+/CXOUxyJLouyLnzcdzkYLGPWVTqHqQrHW5Jns5EOr03b/kNpxdEBLkkjKp1+DuMJpuM4+8Mrq9a3tinlh8CwiI7Ne4csuPSouWJxU0c6NmWtbCKq1TNxZxxpE9s0a95psaNQL+5CIcLcr2kE2M8/aPBsMYU42VTJSOoGG05IkP9Ysu3wLjBXZrFrFXyqgHiNVK3fWz8OpGLAIcmJHpsqn3Ud5adixTYGc1j1f+6JqjJGeewbPSX4U9iVn9+XcfIQwvlINJaTyMntnImsjsV20At4/wlGxbcETYOcFcJH+aMzLMIweeyUxJh/Jx7OipXInTog/tsDqAAbvB07jWKv46BVq0wMp88ZqsLd6Sli3NCBs59hOY27/5m9GAsJy4THPXQ83NmiZ++WXPw5KrHKTwRQZ48IdMpCVoQvqURUSYvLAI8AgwJrt2KSeXNvMIeQQnzwyudyWUE3Qk4Eh10W2UYMio5iEd4Gmk1vS5wERE8z7EERMm3g1bmA0gEyUhHnO4tFDVuEQ7QHYOAlg2a0sAmmHpxRZNyBS4t2YMVuluM1DOLMYozPbql2YB4Q9SQ6SDyN+wwm4QnyqLK4xnj0k175KQSuSC7vvRjVFUb5yr5ztDuGbbnatqDT1KdH7XMfQ/By9zl00UtOaMWVPwApSyFdm7cW0Suu/gqRRlcLboe2PrB90JafqEXVlabYHCHZ4QK0PhFi9QlpdodMK3D8UjTr27a2yJeva8Wude+WbeuCtqV2LTgDo+9HAgCc3nIYvPpizA2OgRxdOyxkE/Mpojw2xWBHdOGvQUUfeBjt/8erppOkXOENpeVXb5/DU5EFSAEIkaXUBm4OBF3+WRCdy1w1+QCJ90VjVYfJ0hYxLXNlOtN/sIFlI7937t+27ASnSRAebo5V8o5QthuLcjbyFF/k2RUsKMw1KuwTs8MpL9+23ism5bDNpz1AhBYvB4jVMzMDjZKCh12tT6U2wB2BpnR7Ralufs6JQFRQUqoAFpTQ863ISg8sEKorKuU4pM40pCllrvp9InnRspiQ6n1YI1RSuOupjxeLdSRCmJ3tKiMZ6XMb49D62bObujfh/3rq+gF0Fl7gHIkZQSOgN8XqD4IpkAitfGbPKizZxr4kd3OI6bHiJc6CT/orKRJd2jqIR2P+gSG4JXyXzw1XTkxSbFYuUBi2ySIUpmC1hsXutOzFn/h+XWAELH+JV78P+sx7HYLh4T9Cg7IftDlYo5C4ybLWYD0PxHcSQFlJIWmM3yKy7OZFm+YY3DT5n+Qjy6ID/WCD7vY5B94VTJRNeDT/oQ0OIdAPqpau08UEA8/8VfOS0WsxwGAErg9utupmFXkWvFw4+h+0DWUv4b2mFJ7E6FlKgBJoLQ1ksvh6eZEtQygy1e+mvaBetWrVTtzG6UQtvS8i6bvXhGGV8/0SJf0LLxXuf0mPtW6QtDwTzXpCtGktQgSn2yqndND0rKMqRvFbOj2ol/3lIl8fXqu26uSMZTuCHBU7Xc8l19IR7nRC92nXI0NMyny4WcqB6r5BrYu/frYy0VEXDztmJ1/WalOSjFC89+B97XFkcYH+bgR7zeWE83qtZE8IlI4SROwEnK9tA0qddHE1fVPVJ3alkWXBU35xG9Tex9pPGGSggjaPrDRusa2rZbGN8OKbqagpkapbeaXWEV6ltRE73xB0VKCooIRUOcrC9IJG+U7dh40WRIHFD3MOM7gXP0Qj9lZrdj49t3kvVyeckQPScExhkiu4GoHh9lHC3wAAhFzWO1EMOo/fYCuClPviCA09uv9JXgmY02n8LxcXBpkczSAJDqnJtjS8ij5YwqtB3Zb9NucEUG/+8JVISVYW8NOUhwOXU/AoDCE6SYnmfWo32gs8G4Hb0csRxqPndFiJKd0TC0HE3CGOzO6KVS/XMLxsAY539zfdUsfCBTcm18UevCU4gQs6UL22dBY/fxrfb8mD75E8LdPq7dyp8PoPOciFk2wcNdIR1m1bq29KQw+Fq2YRn4f6hzkP2jR/B0q3vjjbzMKk2a5PCRt+9ZyzBOz14oM0urAoKAZglZDWe5i+USqqxtLwWrzwBXJOiUfcXYAzLvLbq5q91E9VFwXq9M24XTPLVPalZfErAxfVlVC0UXTT5a/pEPdsV6Xf+DpM3/fVmXNfVh6nscAPD/gBacJAE9E7ZAtMANkdMDaqofNkRGpKBSQb4mogRhF9ICPfadWfWobwvF59ZodjBZINh5LJcC15ohaHwpImAn1f8vsd96FN09eUE2jku6ojNIUku0C1Q3HRpyVzBuxz92CjhhLOGV5WPOa4scfbxM3M30irG8BW+fvCW/OAKt+XuNTwx7ZYJOAnBzaVC8VHHJhx4ADdoK+5qe8iVYdHV8yGUOG9tfLvVtgtqq0Zc8fs/UDU9KpJEqK9Y2wZ4WFbYfwd/Zn0PwBXJgIsKawrtLDmQ6pF9R+lVw3+9ENr8tHZfgA4IKFh89xPyB0Q6hV7UOWnwdplBpii1encfY65O18m7DgJSwwp7G1GwgQUgNhPgcfLqCwytuCDYlmqwMFTlEt5tScC5vE6Glx3Of8MlndxiIAhWw5Hge09Zo1Mx43JcdhNKsEVbH6Rhg17VBWOjfQrghPL5Hgd/wXs7MzzdeyAP00jSL//K7ngTzgfu9mj48zARDNCg1xIlW0otQHW8oC8BPirlzmHEeUHUci8b6yLF3gUtgLwaPK1rb5w/cKt3M0KdedJpw/MuHT5dry0XxFpDCYl2X3Klw7hr9I5dXSrnekZEbloy6sZYjLK8GLdwaewarfZQUGmkXIv0Q/0MT4jxNIh0hN6Zzlh2jFd3adk2dsXZq0Lf1Fj1xt4tnA8SIE6XRt98EmaxSYe8BgxqGAvl1uOPDRFKDHcQhgKhmJVMwRfky6g5hPMf/Ecpy0qheaJNmgee+JTyqEWSwTcj6E35SNiGNtZML1WPojox1mqqwsrbHT0NWqzVIaLlIY6IbZtWmGyq5O94SifU6ZIl8nlAF32gNkA4jmwYPjyfvurTCoYwmGJJGLyPs0+nbc0AI5RJU/NGklwyNxbzLByswFeJEI2RIAzFZvGG6Rr7BMCsuzYbzSr16iPWNTxna+lyaokz7wnl6pt5GF2fJBaG4PWl8H4nxDrbv0AKt6ojNS0rk9TNKfyqcxAkebaIp14N3MGJaaFryKkySQKVOQlAjSwRA1JA0lPfp3F4b0rcs3V90Xfh6mFbZiTSQzKFndmL0g8PLIzexJ3G9/ZS4dKUiptzRt9dXZjN9AWkJicz/LG0HNMevwlqrk5LTtcrWyY1Wh16C8va+9cq8CHTW8wBmx2HOihVL/YJz3OuZfJYzWiDjl7xRhvhlrtPdsf3cjfozHX1TM0ffZXfpuKdwYJAtiMGtcp3GWcdNAt4pg822gM9TXFyJHdzt/bN1g25KuGN85SIZhzKHbgQhRp9KI7Yz48t4kjvGX6fhAlpXLDNMqJW2oSmq8e2Noq8tsTtBWsmu3w6yGxVc5rTxcSYQyZTZUrIwmpXub9x40vUXZcdHmUdSDvMA1QpfbbkP/YSMJlzlyuBcb0bhO4c9+XJb9FTb0NTOkbvPu0gU2fuwfWUnQ9GOUXILtlYoDB6Y7uncuzqoiSGy3eWu5AqG8p6TQUS3Qo5256ZonsPVL2RBrJZteSnje9BXjcelWQei+ILuu6xpunzotUk5Y28MCrYPAWoI4NaVe+lHfqoW0aH9EyGopT46omoXgnlgCSO66ZMa/hcG/mXyN687PClKomTAx4iWECTWnMk9xA+oRjVZKvxF1k8aD7eqUJpzdUbGV34ULt06vbeI1iILYWdxnUExfeqyJKSBhoSAQev1PBOh1qbU1fhgcqSlTfn5fHkANKG4T6Ne+PmNwbvmjosbsF7uFn1ecmX7UX3XT1qW306pA44vKLsE9WqsDA1S46alnJv1sxynJFdL4Qge62n9jXB0hzC/BqX2/grNjcrbSoZWemWMFuRYFUKzhqx3WbSjj4ruMbLbF6Gt9EhldGIA6HgsYXbiHNLQTxyaiP+6bfjrxCQzP3lUvv13stS8wLkX5ZUKChGH2bdlrZNe2fBE/xgxUUk+Ir8smn6zzJ2O5I0Tzj/rVn8usBGUOhM/eX6u1cHpMjSw63CIKB6VTKmQqxCMyUHn13TjuAIwGQotDo6O18kH55lyIVAWxpwA4xn815xrn22SPiSu5tHSWsxoMVuGVN1ohFMOWQIBMIGIZ4/NSQ0cWhHUZzTRcii8gjKMzEHe9eRruoxb2tcVfyb4JSpv+pAaqRJVlPwYZ7FI0eMkG9EnF70Auctk7uHNbbbfIUekdV9cNwjbyiJWrAB0wWObsEbK/1iVxmWravMmPsysBJXCoBasWCfBpvREMq6bn97R5il9Q9/kzYj40GGTOBcn8fZGcLWn+0iwnixB+iYjE/jLSUGQeHVYQl4rfVusP+QlvTviPbmU+cmM2YSByZozO3PcCHAMW0DcJeQfGM8yZbUkD12uPR+Me8arGyb5XXARpJH4ZbkmxaF7dYaER2iD1gMzhFT8oDl1e0XhNurAeVuCdZOqGw3FDfiq6cYJGyfUBs/9T2OwnN3mnYVbEEKnKE1oLlg9yCriyewoxncYo6CZzwfX7XAyBVz4BNeV7DojxsLSqj5gtAZOAlvRXKE7fwZMUcWrceARjBf+MF0RwAKOoM+QjFBnDunBUa9lSORF6Mgx/Tvyy7WI6TGwHjJ8syrLGQ25j0KtHph20+X+/n60noSfDMfhpyntkPwIZ/8AE6Cq+ky4sTUI+PyqqkIg1wtcb/qJeIouLQjxXA5/f3RkFtOcAmdBhJMiNr03pfIIJidXRYFHBUqL2MwiSTvLHLWJQ0h4/7C/bY+bjhzhkbyE+p3ZMvlpnYJgdoVYtQZrEAs/0EefwrPQNsjmflHORD4RcpMHbnYkEBgnF6f2gQ+FFAtvkRUAKapkJcMEc2YfLQIybVJ5BUEBBbirccID2KuE9drzXkteRo4N8/oCunhfH9ppvG3/FFCloiN6B+bWkZzAcZ9Ihb/GzBC98KDSIdzFtu4RiXYglVJrKZpv87qaM+gXBBszL0SrgkFZShoLqZnedvfY1/I3B2w+J6XZJp5YWArSPqPoMhq/9yFOhUvWZoMXfL5q6bulbtGGCU+0nrT1WJu8/6K9aFJZx+2tQZcECAz/RASNTZNj/J0JJQqSxCiQaiVHpxvhseLxMPz1iousVnb1M8VXeWWqG3YW6gnqMO/7+9lK0xlvPNDtNT0aRNLL1Agiu7uYUD/j/gPsTPlWA5daoWcqVJuRtsEBdE+A/2P4Ol2Nhkvd33J9YybP6xqP7DrPxgRA32JfTHJ+lvqZpAbrBfb6cu8VHbVoXXBWgDOJmnyML0x6T7cV61RxGtINZ9CrkIbmaxTUN4G7tPLmbQNzsKmMirGFY8LUs1U2Yz4GIUknlv5FppF2vJ851/+aJHcRGwD2oI4u77hofb4RwNBWdYAGetSK7qkTuFhyk5SP0UfJGkkUpjfzJp6HzTe8mXR4PWJW9Pkq+BZFrMbcnDEataiAMhuXQ8lmLPw0N8oZ8BI7XgdpsfcqAM0CwgtC6URs5d9yDEB7w4zDMvMy0QYvekCBeP6QumHfw0Wj0IqJrxFNoVGYANia/u2jr9P+u1jOK+NIu4XWM6Fk2qb1Zz3mUcS7/gw6qK9tdExiWy8mbNhuOxHM3pD3FxSqjoHAD+J92Out/7zj5CnNMU6yNXGopPIWVjPw7ZlQQXcMjjHfbdutHTDkUhtSMEbS6A0KP41UHaOlu5CgBXS8Ew0xYRzkihqnEQM5UtvoUjR9FI7XQXQ8BMRfIFCq6Y1AlatH4+RS9Uf9UGYHSsUgeg1dXMjo0UGpv+dIKUpHELeoPSaEvIvFqgeIEBM/WWwRhb2nEjgmVb3t/n2R6Kp6n6OxrSaJzWRgrLAvIvuhjB4yvC7AXsNdMsTILIrA+Wo5+XKDIxAXrXNx6XHACk9dfRdba6FJB2p5fNiw+aOCuruZoD+I/tFwwPhWnm/CO+BGk20fBundzSh4n2qoGxCNmZPHaz7zDAK38q9jGkRcZuYIkuPQtzq/CiWV2OMvazbcfA3IsTNlbDLJ8NWiD+9sB0HM64GfWzJnDNmKOW/3VBd+oK1dKURhmfz4ooAs84Wi66KJeAFq0dvSNuhdH6m/Y+9d+97yIxCZXMhTZHIcT1DAFtf6f3dxaBjfC3nRi/HoVfochpcoGrdvBk9uYhB5jsDs0i08a/M+NeGcgTopqN+vqUnZfScgGyIsfvlku+Z3Sv00tgHePwmXogfBLYq7fM23bqFcz8FJFC1EgA3xG2fljdB6NAWi1aTb+8k3/4tUlPcmRdAlnAzRY/6RJ0n+FehCcED9N0vNAzX0JZms2c964tAxzo5TAv+7s6t2nLg1Qzh1pXVbAi5PepQmfttSkdAmW0GCyD+g/m0J6FpTYHLSXoewCZWqBWOOwLBbzB5KQFdKAqgE8zmQYuHoGmeq/8WJFXSCxjavBj5+/fGPDJ3W3kNVwH/AckYltlaRQLUUPQ+vTqcBSUbVljuDCXQ4G741bKZJjoKt9lJGxQbjsG+wLoo1efZa7D0CsDRmaAz+y0Z4lFm/Jf2OmNAiABFsPob8CqnVCF4S1OmN2qSY540mYSWKbI5IG/RwLb8osH4VMXrJY1VEHGcdzjD7PXsIQ65pOe4UnuN9VXMhHJfpNvZLYnWK18soVdGSSewrk3Pcd2wCOrwnDVn0wuMDToK9r5AZmoK9V5zS+MpAblbMMKRxQR6vkvF87BnHSUdkR1Ee89cE7WRuUOMqrArcrsqZn4Op1f8Dmk4tzUVyC5ml5onUJ+WQtBGzeHLVM8AFJiDDvHkO5MlrmBF3TxvTeaEf0G/clHbDXEYFcst+IdqQfydpnjyuZAZv8PtfKYWLOZamN7KEU+Po09QXWxJNxWDc3i8tFTMb3eNSRRySVDkqTEfO/My6k0D0xaISOVFIRkk8nzT5PDoTl/HPbrd0MpqUB83gaMrTwzSLxpDMNn8CQYkmhinltp0WcLxvVVakNjIch1Fg1JloWDe7IbcACXoW9vzV9cXjDGivdNsjQoxZi3upNyhOnC7oa5ZKPZLyxUqvUaaqE9dQvtQIA7ME50CgS6LmTa25Nonx1ifq6+rvABuElBbOLReMIIhbM5EdV9QisinYyJ6POgexEdcrezu+vsqIEz6qI80+DVagac3eoPCtK1PTIqoXoTb3jPWD+4BOAa7C4AwNNQDAlhHYhRduWf2k7QiNqW/boIbLkrhj2FuxsERcKyNltfUud2DgBV8IC3u5cPv52nWRKa0uCviwl4/3BQwA/NocjpsPYCvPmrmcW/qlXSyUwriI6j/dZkx0YhtqQ9h5y63PwGwRwFkV+ltb7ep0mj9gKlNIevhcM0P55Mv+gbAI0o9n7pqa2r45X8yZOU7dYeXhTzoVGM5YuFIFgXOqT4Is+MmywZCQxKUs0OGVwgeAz0QT6UmLKbQ2eRvu8pj/f9CD7sar5cVV+X2dzuK+5Ybv8Hnahn2zKhqi2l6e0oEfNXYV7dOKbJk2qDIjch0YkvHS/+spbS+h3cklZOp4H2IyNzj8y78iwJQqHp4q9XWMIbD1fY6mdTl21Qmv9TBw7bYT5kT0YcaDd0DNeiDlmGQb5iJOKQpG9UF/5zmgcx+3MQjPyKTkEOAgq+fvs+txW7iYnvhLOAiPB0jrQqFLiHG9yUXb7jDR7WN0NUb/vjKoYag2hLET5wqqyFzZm9t7/vJZh+5ZmV6sooIAcEUs6/BgYVjl1jwsn+8WVws3g4XbJJ312XcIGckUmNPKehb+s8cYuFmM3r+79MAUlerSHBo61M7lBdCtXWlSVP9o0+TWKTClfKeDAY6R0M50cUuYpPfzxtM2rwsWlOa10GtuBLYQaSHxaIeIjd/oGc/P5bd5bDxE5V+06ddvvRfbSMHwYGh+ezRnP4tUCLgmj2mEH5EV5+0C2GX2v6HLSGa9tHUEcLRGY3VR9BOxIbPf5+Lq3Mlc1ukFRtwrdA2C05IcT1lZgRCzCCqcBAnwEi6gy9yt9cZPVuBWdHoVQae0gXK9icwju7VDUTPUG9XTJ/4TPULVirKMi6xbhgsMw5JwsNzd9UxovKFvREz5ezLaXSlywQFhyCyGJvCjKjSwO6mgFIfHR9PrJ+D05k4oEHUxRHDpvVfJEoKE7SkUDXMSu/TCZyGXEOgZZrYIdoZrJDAKxcLyfYati5vgxtGds5XFyY/M3lDGfGdsadXIDMa5QqE81G5D7i8TVVPHXx7oQ7PVTJ2lZoM/BSHoVtzJI9g9PxH3HyQYyzG1MALL4VojZJF3V+mGqjXKsGXNqUn2+Hwo7P4EctbJ/ZQN5G2bHhG9KGNtggjs3KVW7Pn7LR/yfR+yPPZkNmr7vWhZ3amBTI1bQojjCV6D07yt+TRaZScGUJLdYqFq5mvfBYuh0SB98NYXLeInZamjm9l/E0/2mQAvjS5DxpWt+g7De2HOcKMKa4tvIJXBAfhHzLeaw9Y6qesvtgldB5reygOf2WF6MmY2SuUZOCx15zRuJU1jMQmbW+8JRjtXG7PgXFm0lxT7H7GFnJ/EvZj+9oUgSkqU68t3SgXfpb4GC/q/RWejpCHNESIINyNXb+P1QGpaHdV7swYdHqc7v6vIEderfPzj8u02DD/NQ+NU3fx2lS6fCWGRMYzXV3RMktWhxQ1IouKcX+umpKEz8YyEHLdpD0SrQOYNAKdraJ3eZVkx/pSq0tOMQmOyczBDfNQu1v2+Bufk3XffOeFL4EAuHpAKkXo3HjH7+bQEHsOmldiLVYI5QM6d+4J8vnwwqOq1wVVgDQvv4hOpcEiNCEoqUE6+E/p9SdnpJ++p+oSYmi/FA82kVlJWHDDW/Rh6Y2d1PbBHdUmi+Kb50wXLB+ALEuGahmu5PWOVeTWrx3XPnGTrWj9FzvmZWQ7Dgo5//sAQH/TWcA+XVZAFrPZ+WgyttnjzxdfecXnN7CSJwJyeSp7w5CpiuJHcFgZTvENkDUoTptI8IKCjJviH2dy8VHosZqHWEdTcdAheZtWDgMRj6JV+9ax43FU+GYHu7YLowN4kuSoYO05xewB+C6C56oL6MmYGMbw4lwoubt8HLkd0bqpsN4DAbOrf7nX+n2W4X81hLIhrFymw+aTBOMTBM7pAoOZIe0U8G27MhmiPb7MCFOV6UMgwhJv1IBFZv6HInPnpMBUDNQOe0dLv72g9LF6SK3NKiXk6jkni/WBhMz63DK7a3Y071vBgfrrG+tNBXCuYdqsJYpn1LRe84kHrcYGwb5uvaGms9tA84kbZ1jErDLKmByODXFvQxDOYhRU0eeqlYuASHb9OjygMvzLOaJC7xV2Aj0ugY1uqy0XONnLdlXdhx4CR8+k3DzGvGZXXIp1a1ncSgxNsmijUcHJOgPHag4TGUyQwlQMtKK+F+NVJqSajfAHhhLA4ehEw3a9xTNf4vAmr5L//LdujG0rRe8QK/PmQ3kca61Si+0KsplgQFJkBdzNSxjxlG0H5jBgf8VjqoHRru5bsOKd0/M11YXQkqPwvlwnEqYLiHJ/XmnKHJZQ4aZz5C6ih9poaq7/xk9z4Mm9x4c78r+B6pkxew7JXt+APgf8FInGVSvAAHkOp3sAgBRX/V4scRn+wIAAAAABFla